# E3-K — Sensibilité de la connectivité du GCN (k = 1…6)

**Objectif scientifique** : tester le GCN **principal à deux couches** de l'article, avec les mêmes données et paramètres figés, en faisant varier **uniquement la topologie** déterminée par `k` plus proches voisins (corrélations d'entraînement). Cette analyse est **distincte** des architectures diagnostiques H1–H5 / E3-I.

**Données** : `outputs(1).zip` ou dossier `outputs/` extrait. La construction de nouvelles données n'est pas nécessaire : les tenseurs normalisés et cibles de `07D_final_alignment` sont réutilisés directement. Le notebook vérifie d'abord l'inférence d'un checkpoint original `k=2`, puis calcule les résultats de tous les entraînements et exporte les CSV pour le tableau de l'article.

**Attention méthodologique** : les poids et les opérations d'**inférence** de l'architecture originale sont reproduits et vérifiés. Les détails précis de l'ordre des opérations *dropout* pendant l'entraînement n'étant pas fournis par l'archive, la procédure d'entraînement ci-dessous utilise une convention explicitement documentée. Comparer obligatoirement le résultat du nouvel entraînement `k=2` au résultat original avant toute revendication de reproduction stricte. Le test reste interprétable comme **expérience complémentaire homogène** si les 18 entraînements utilisent exactement la même implémentation.


In [ ]:
# Exécuter une fois dans votre environnement si nécessaire :
# %pip install numpy pandas torch matplotlib nbformat

from pathlib import Path
import io, zipfile, json, random, copy, hashlib, time, platform
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Modifier ce chemin si outputs(1).zip n'est pas placé à côté du notebook.
OUTPUTS_SOURCE = Path("outputs(1).zip")
# Ou : OUTPUTS_SOURCE = Path(r"C:\...\outputs")  # dossier extrait
RESULT_DIR = Path("E3_K_sensitivity_results")
RESULT_DIR.mkdir(exist_ok=True, parents=True)

K_VALUES = [1, 2, 3, 4, 5, 6]
SEEDS = [42, 123, 2026]
HIDDEN = 64
DROPOUT = 0.1
BATCH_SIZE = 256
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-5
MAX_EPOCHS = 80
PATIENCE = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# CPU est plus proche de l'environnement original (PyTorch CPU).
# Remplacer par torch.device("cpu") si la reproductibilité prime sur la vitesse.

RUN_TRAINING = True
print("Device:", DEVICE, "| Torch:", torch.__version__)


## 1. Accès aux données et contrôle de cohérence

Les tenseurs ont les dimensions `(snapshots, 5 instants, 7 dispositifs, 3 variables)` et les cibles `(snapshots, 7 dispositifs)`. Les 13 790 cibles de test correspondent à **1 970 snapshots × 7 dispositifs**.

In [ ]:
from pathlib import Path
import zipfile
import io
import json
import numpy as np
import pandas as pd

# ============================================================
# 1. Configuration du chemin vers les données
# ============================================================

OUTPUTS_SOURCE = Path("outputs") if Path("outputs").is_dir() else Path("outputs(1).zip")

SEEDS = [42, 123, 2026]

print("Dossier recherché :", OUTPUTS_SOURCE)
print("Existe :", OUTPUTS_SOURCE.exists())

assert OUTPUTS_SOURCE.exists(), (
    f"Dossier introuvable : {OUTPUTS_SOURCE}"
)


# ============================================================
# 2. Lecture des fichiers depuis outputs ou une archive ZIP
# ============================================================

class OutputsReader:
    def __init__(self, source):
        self.source = Path(source)

        if not self.source.exists():
            raise FileNotFoundError(
                f"Introuvable : {self.source.resolve()}"
            )

        self.z = (
            zipfile.ZipFile(self.source)
            if self.source.is_file()
            else None
        )

    def read(self, relative_path):

        if self.z is not None:
            p = "outputs/" + relative_path.lstrip("/")
            return self.z.read(p)

        # Lecture directe depuis le dossier outputs
        root = self.source

        local = (
            root if root.name == "outputs"
            else root / "outputs"
        ) / relative_path

        if not local.exists():
            raise FileNotFoundError(
                f"Fichier introuvable : {local}"
            )

        return local.read_bytes()

    def csv(self, relative_path, **kwargs):
        return pd.read_csv(
            io.BytesIO(self.read(relative_path)),
            **kwargs
        )

    def npy(self, relative_path):
        return np.load(
            io.BytesIO(self.read(relative_path)),
            allow_pickle=False
        )

    def json(self, relative_path):
        return json.loads(
            self.read(relative_path).decode("utf-8")
        )


# ============================================================
# 3. Chargement des données GCN
# ============================================================

reader = OutputsReader(OUTPUTS_SOURCE)

ROOT = "07D_final_alignment/"

X, y, meta = {}, {}, {}

for split in ["train", "validation", "test"]:

    X[split] = reader.npy(
        ROOT + f"X_graph_{split}_scaled.npy"
    ).astype(np.float32)

    y[split] = reader.npy(
        ROOT + f"y_graph_{split}.npy"
    ).astype(np.float32)

    meta[split] = reader.csv(
        ROOT + f"metadata_graph_{split}.csv"
    )

    assert X[split].shape[1:] == (5, 7, 3)

    assert y[split].shape == (
        len(X[split]), 7
    )

    assert len(meta[split]) == len(X[split])

    print(
        split,
        X[split].shape,
        y[split].shape,
        "node-level =", y[split].size
    )


# ============================================================
# 4. Vérification des échantillons
# ============================================================

assert [
    y[s].size
    for s in ["train", "validation", "test"]
] == [64701, 13825, 13790]

print("\nDataset alignment validated.")


# ============================================================
# 5. Vérification du protocole original GCN
# ============================================================

protocol = reader.json(
    "08C_gcn_graph_ablation/07_reproducibility_manifest.json"
)

assert protocol["selected_configuration"] == "GCN_h64"

assert protocol["final_training_seeds"] == SEEDS

print("Archive protocol validated.")
print("Données chargées avec succès.")

## 2. Reconstruire les graphes à partir des corrélations du train

La matrice de similarité provient **exclusivement** de l'entraînement. Pour chaque nœud, on retient ses `k` meilleurs voisins, puis on prend l'union des arêtes orientées et on rend le graphe non orienté. Les boucles réflexives sont ajoutées **uniquement** dans le calcul de la convolution, et ne sont pas comptées dans le tableau de densité.

In [ ]:
# ============================================================
# E3-K — Reconstruction des graphes pour k = 1 à 6
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

# 1. Paramètres de l'analyse de sensibilité
K_VALUES = [1, 2, 3, 4, 5, 6]

RESULT_DIR = Path("E3_K_sensitivity_results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# 2. Chargement des dispositifs et de la matrice de similarité
node_table = reader.csv(
    "07C_graph/01_graph_node_index.csv"
).sort_values("node_index")

uuids = node_table["uuid"].tolist()

sim_df = reader.csv(
    "07C_graph/02_train_operational_similarity_matrix.csv",
    index_col=0
)

S = sim_df.loc[uuids, uuids].to_numpy(dtype=np.float64)

assert S.shape == (7, 7)
assert np.allclose(S, S.T, atol=1e-10)

# 3. Construction des arêtes selon k
def make_edges(k, similarities):
    n = len(similarities)
    assert 1 <= k < n

    edges = set()

    for i in range(n):
        # Tri décroissant des corrélations
        # En cas d'égalité : ordre de l'indice du dispositif
        ranked = sorted(
            (j for j in range(n) if j != i),
            key=lambda j: (-float(similarities[i, j]), j)
        )

        for j in ranked[:k]:
            edges.add(tuple(sorted((i, j))))

    return sorted(edges)


def edge_index(edges):
    return np.array(
        [
            (i, j)
            for a, b in edges
            for i, j in ((a, b), (b, a))
        ],
        dtype=np.int64
    ).T


# 4. Reconstruction des six graphes
edge_sets = {
    k: make_edges(k, S)
    for k in K_VALUES
}

# 5. Vérification du graphe original k=2
ref_edges = reader.npy(
    ROOT + "edge_index_operational.npy"
)

assert (
    set(map(tuple, edge_index(edge_sets[2]).T))
    == set(map(tuple, ref_edges.T))
), "ÉCHEC : la topologie k=2 ne correspond pas au graphe opérationnel original !"

# 6. Statistiques des topologies
rows = []

for k, edges in edge_sets.items():
    rows.append({
        "k": k,
        "nodes": 7,
        "edges": len(edges),
        "density_percent": 100 * len(edges) / 21,
        "mean_degree": 2 * len(edges) / 7
    })

topology_table = pd.DataFrame(rows)

display(topology_table)

# 7. Export des statistiques
topology_table.to_csv(
    RESULT_DIR / "01_k_topology_statistics.csv",
    index=False
)

# 8. Export des arêtes
edge_rows = [
    {
        "k": k,
        "node_i": i,
        "node_j": j,
        "uuid_i": uuids[i],
        "uuid_j": uuids[j],
        "train_pearson": float(S[i, j])
    }
    for k, edges in edge_sets.items()
    for i, j in edges
]

pd.DataFrame(edge_rows).to_csv(
    RESULT_DIR / "02_k_topology_edges.csv",
    index=False
)

print("\nContrôle k=2 : graphe opérationnel original retrouvé exactement.")
print("Les six topologies ont été reconstruites.")
print("Résultats enregistrés dans :", RESULT_DIR.resolve())

## 3. GCN principal et vérification indépendante des checkpoints originaux

Entrée GCN = aplatissement des cinq pas temporels **pour chaque dispositif** (`[batch, 7, 15]`). Pour chaque couche : agrégation par l'adjacence symétriquement normalisée avec boucles réflexives, puis couche linéaire et ReLU. La tête fournit une prédiction de température par dispositif.

**Test crucial** : charger un modèle déjà entraîné de l'expérience principale et vérifier que l'inférence donne la RMSE archivée. Cela garantit que la mise en forme des entrées, l'ordre des dimensions, les opérations de convolution et la normalisation correspondent à l'architecture originale en évaluation.

In [ ]:
# ============================================================
# 3. GCN principal et vérification des checkpoints originaux
# ============================================================

import io
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

# Sélection automatique CPU / GPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilisé :", DEVICE)


# ============================================================
# 1. Normalisation de la matrice d'adjacence
# ============================================================

def normalized_adjacency(edges, n=7):
    a = np.eye(n, dtype=np.float32)

    for i, j in edges:
        a[i, j] = a[j, i] = 1.0

    degree = a.sum(axis=1)

    return torch.tensor(
        a / np.sqrt(np.outer(degree, degree)),
        dtype=torch.float32
    )


# ============================================================
# 2. Architecture GCN principale
# ============================================================

class GCNLayer(nn.Module):
    def __init__(self, fin, fout):
        super().__init__()
        self.lin = nn.Linear(fin, fout)

    def forward(self, h, adj):
        return self.lin(torch.matmul(adj, h))


class PrimaryGCN(nn.Module):
    def __init__(self, hidden=64, dropout=0.1):
        super().__init__()

        self.gc1 = GCNLayer(15, hidden)
        self.gc2 = GCNLayer(hidden, hidden)
        self.head = nn.Linear(hidden, 1)
        self.dropout = float(dropout)

    def forward(self, x, adj):

        # x : batch x 5 x 7 x 3 -> batch x 7 x 15
        x = x.permute(0, 2, 1, 3).contiguous().reshape(-1, 7, 15)

        h = F.relu(self.gc1(x, adj))
        h = F.dropout(
            h,
            p=self.dropout,
            training=self.training
        )

        h = F.relu(self.gc2(h, adj))
        h = F.dropout(
            h,
            p=self.dropout,
            training=self.training
        )

        return self.head(h).squeeze(-1)


# ============================================================
# 3. Prédiction
# ============================================================

@torch.no_grad()
def predict(model, inputs, adj, batch_size=1024):

    model.eval()
    out = []

    for start in range(0, len(inputs), batch_size):

        b = torch.from_numpy(
            inputs[start:start + batch_size]
        ).to(DEVICE)

        out.append(
            model(b, adj).cpu().numpy()
        )

    return np.concatenate(out)


# ============================================================
# 4. Métriques
# ============================================================

def metrics(pred, target):

    e = (
        pred.astype(np.float64)
        - target.astype(np.float64)
    )

    return {
        "RMSE": float(np.sqrt(np.mean(e**2))),
        "MAE": float(np.mean(np.abs(e))),
        "Bias": float(np.mean(e)),
        "Error_SD": float(np.std(e)),
        "R2": float(
            1 - np.sum(e**2) /
            np.sum(
                (
                    target.astype(np.float64)
                    - np.mean(target)
                )**2
            )
        )
    }


# ============================================================
# 5. Vérification des trois checkpoints originaux (k=2)
# ============================================================

original_metrics = reader.csv(
    "08C_gcn_graph_ablation/"
    "02_metrics_all_topologies_seeds_splits.csv"
)

A2 = normalized_adjacency(edge_sets[2]).to(DEVICE)

checks = []

for seed in SEEDS:

    b = reader.read(
        f"08C_gcn_graph_ablation/"
        f"G_operational_GCN_h64_seed{seed}.pt"
    )

    state = torch.load(
        io.BytesIO(b),
        map_location="cpu",
        weights_only=True
    )

    checkpoint = PrimaryGCN().to(DEVICE)

    checkpoint.load_state_dict(
        state,
        strict=True
    )

    calc = metrics(
        predict(checkpoint, X["test"], A2),
        y["test"]
    )["RMSE"]

    expected = float(
        original_metrics.query(
            "topology == 'G_operational' "
            "and seed == @seed "
            "and split == 'test'"
        )["RMSE"].iloc[0]
    )

    checks.append({
        "seed": seed,
        "checkpoint_RMSE": calc,
        "archived_RMSE": expected,
        "difference": calc - expected
    })


# ============================================================
# 6. Affichage et validation
# ============================================================

checks_df = pd.DataFrame(checks)

display(checks_df)

assert (
    checks_df["difference"].abs() < 5e-5
).all(), (
    "Le modèle n'est PAS conforme à "
    "l'inférence du checkpoint original."
)

print(
    "\nContrôle d'inférence : "
    "conforme aux trois checkpoints k=2 originaux."
)

## 4. Entraînements k = 1…6, graines 42/123/2026

Hyperparamètres du protocole figé : hidden = 64, dropout = 0.1, Adam (`lr=0.001`, `weight_decay=1e-5`), batch = 256, au plus 80 epochs, early stopping après 10 epochs sans amélioration de la **RMSE de validation**. Le test n'est **jamais** utilisé pour choisir l'epoch.

La convention de dropout retenue ici (après les deux ReLU) reproduit l'inférence du modèle original. L'archive ne documente pas explicitement le masque pendant l'entraînement : le nouvel entraînement `k=2` sert de contrôle empirique. Les résultats peuvent dépendre de la version PyTorch, du CPU/GPU et du parallélisme.


In [ ]:
# ============================================================
# E3-K — Entraînement GCN : sensibilité au paramètre k
# ============================================================

import random
import time
import copy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

# ============================================================
# 1. CONFIGURATION
# ============================================================

K_VALUES = [1, 2, 3, 4, 5, 6]
SEEDS = [42, 123, 2026]

HIDDEN = 64
DROPOUT = 0.1
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-5
BATCH_SIZE = 256
MAX_EPOCHS = 80
PATIENCE = 10

# Activer l'entraînement
RUN_TRAINING = True

# IMPORTANT :
# k=2 a déjà été entraîné avec les trois graines.
# Nous entraînons uniquement les autres configurations.
run_order = [1, 3, 4, 5, 6]

# Dossier de résultats
RESULT_DIR = Path("E3_K_sensitivity_results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

# Fichier cumulatif des métriques
results_file = RESULT_DIR / "03_per_seed_all_splits.csv"

print("=" * 60)
print("E3-K — ANALYSE DE SENSIBILITÉ DU GCN")
print("=" * 60)

print("Device :", DEVICE)
print("Valeurs de k à entraîner :", run_order)
print("Graines :", SEEDS)
print("Nombre maximal d'entraînements :", len(run_order) * len(SEEDS))


# ============================================================
# 2. REPRODUCTIBILITÉ
# ============================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ============================================================
# 3. ENTRAÎNEMENT DU GCN
# ============================================================

def fit_gcn(k, seed):

    set_seed(seed)

    # Graphe correspondant à la valeur de k
    adj = normalized_adjacency(
        edge_sets[k]
    ).to(DEVICE)

    # Architecture GCN principale
    model = PrimaryGCN(
        hidden=HIDDEN,
        dropout=DROPOUT
    ).to(DEVICE)

    # Optimiseur
    opt = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    # Données d'entraînement
    x_train = torch.from_numpy(X["train"])
    y_train = torch.from_numpy(y["train"])

    # Données de validation
    x_val = torch.from_numpy(
        X["validation"]
    ).to(DEVICE)

    y_val = torch.from_numpy(
        y["validation"]
    ).to(DEVICE)

    best_rmse = float("inf")
    best_epoch = 0
    best_state = None

    bad_epochs = 0
    history = []

    started = time.time()

    # ========================================================
    # BOUCLE D'ENTRAÎNEMENT
    # ========================================================

    for epoch in range(1, MAX_EPOCHS + 1):

        model.train()

        # Mélange des snapshots d'entraînement uniquement
        order = torch.randperm(len(x_train))

        squared_sum = 0.0
        count = 0

        for ids in order.split(BATCH_SIZE):

            bx = x_train[ids].to(DEVICE)
            by = y_train[ids].to(DEVICE)

            opt.zero_grad(set_to_none=True)

            # Prédiction
            yhat = model(bx, adj)

            # Fonction de perte
            loss = F.mse_loss(yhat, by)

            # Rétropropagation
            loss.backward()

            # Mise à jour des poids
            opt.step()

            squared_sum += float(loss.item()) * by.numel()
            count += by.numel()

        # ====================================================
        # VALIDATION
        # ====================================================

        model.eval()

        with torch.no_grad():

            squared_error_sum = 0.0

            for lo in range(0, len(x_val), 1024):

                p = model(
                    x_val[lo:lo + 1024],
                    adj
                )

                squared_error_sum += (
                    (p - y_val[lo:lo + 1024]) ** 2
                ).sum().item()

            val_rmse = float(
                np.sqrt(
                    squared_error_sum / y_val.numel()
                )
            )

        history.append({
            "k": k,
            "seed": seed,
            "epoch": epoch,
            "train_MSE": squared_sum / count,
            "validation_RMSE": val_rmse
        })

        # ====================================================
        # EARLY STOPPING
        # ====================================================

        if val_rmse < best_rmse - 1e-10:

            best_rmse = val_rmse
            best_epoch = epoch

            best_state = copy.deepcopy(
                model.state_dict()
            )

            bad_epochs = 0

        else:
            bad_epochs += 1

        if bad_epochs >= PATIENCE:
            break

    # ========================================================
    # RESTAURATION DU MEILLEUR MODÈLE
    # ========================================================

    if best_state is None:
        raise RuntimeError(
            f"Aucun modèle valide pour k={k}, seed={seed}"
        )

    model.load_state_dict(best_state)

    # Sauvegarde des poids
    torch.save(
        model.state_dict(),
        RESULT_DIR / f"GCN_k{k}_seed{seed}.pt"
    )

    # Sauvegarde de l'historique d'entraînement
    pd.DataFrame(history).to_csv(
        RESULT_DIR / f"history_k{k}_seed{seed}.csv",
        index=False
    )

    # ========================================================
    # ÉVALUATION TRAIN / VALIDATION / TEST
    # ========================================================

    results = []

    for split in ["train", "validation", "test"]:

        predictions = predict(
            model,
            X[split],
            adj
        )

        target = y[split]

        model_metrics = metrics(
            predictions,
            target
        )

        results.append({
            "k": k,
            "seed": seed,
            "split": split,
            "n": target.size,
            "edges": len(edge_sets[k]),
            "best_epoch": best_epoch,
            "best_validation_RMSE": best_rmse,
            "seconds": time.time() - started,
            **model_metrics
        })

        # ====================================================
        # SAUVEGARDE DES PRÉDICTIONS DE TEST
        # ====================================================

        if split == "test":

            timestamps = np.repeat(
                meta[split]["target_timestamp"]
                .astype(str)
                .to_numpy(),
                7
            )

            predictions_df = pd.DataFrame({
                "target_timestamp": timestamps,
                "node_index": np.tile(
                    np.arange(7),
                    len(predictions)
                ),
                "k": k,
                "seed": seed,
                "y_true": target.reshape(-1),
                "y_pred": predictions.reshape(-1)
            })

            predictions_df.to_csv(
                RESULT_DIR /
                f"predictions_k{k}_seed{seed}.csv",
                index=False
            )

    return results


# ============================================================
# 4. CHARGEMENT DES RÉSULTATS EXISTANTS
# ============================================================

metrics_rows = []

if results_file.exists():

    existing_df = pd.read_csv(results_file)

    # Éliminer d'éventuels doublons issus de relances précédentes
    existing_df = existing_df.drop_duplicates(
        subset=["k", "seed", "split"],
        keep="last"
    )

    metrics_rows = existing_df.to_dict("records")

    print("\nRésultats existants chargés :", len(metrics_rows), "lignes")

else:
    print("\nAucun résultat antérieur trouvé.")


# ============================================================
# 5. VÉRIFICATION DES CONFIGURATIONS DÉJÀ TERMINÉES
# ============================================================

completed_pairs = set()

if metrics_rows:

    existing_results = pd.DataFrame(metrics_rows)

    for (k, seed), group in existing_results.groupby(
        ["k", "seed"]
    ):

        available_splits = set(group["split"].tolist())

        if available_splits == {
            "train",
            "validation",
            "test"
        }:

            completed_pairs.add(
                (int(k), int(seed))
            )

print("Combinaisons déjà terminées :", len(completed_pairs))


# ============================================================
# 6. LANCEMENT DES ENTRAÎNEMENTS RESTANTS
# ============================================================

if RUN_TRAINING:

    for k in run_order:

        for seed in SEEDS:

            # Ne pas réentraîner une combinaison déjà terminée
            if (k, seed) in completed_pairs:

                print(
                    f"\nDéjà terminé : k={k}, seed={seed} — ignoré."
                )

                continue

            print(
                f"\nEntraînement k={k}, seed={seed}...",
                flush=True
            )

            # Entraînement et évaluation
            results = fit_gcn(k, seed)

            # Retirer les anciennes lignes de ce couple, si présentes
            metrics_rows = [
                row for row in metrics_rows
                if not (
                    int(row["k"]) == k
                    and int(row["seed"]) == seed
                )
            ]

            # Ajouter les nouvelles métriques
            metrics_rows.extend(results)

            # Sauvegarde immédiate après chaque entraînement
            pd.DataFrame(metrics_rows).to_csv(
                results_file,
                index=False
            )

            completed_pairs.add((k, seed))

            test_row = next(
                row for row in results
                if row["split"] == "test"
            )

            print(
                f"Terminé : k={k}, seed={seed}",
                f"| Test RMSE = {test_row['RMSE']:.6f} °C",
                f"| Best epoch = {test_row['best_epoch']}",
                flush=True
            )


# ============================================================
# 7. BILAN FINAL
# ============================================================

final_df = pd.DataFrame(metrics_rows)

if not final_df.empty:

    completed_test = final_df[
        final_df["split"] == "test"
    ].drop_duplicates(
        subset=["k", "seed"],
        keep="last"
    )

    print("\n" + "=" * 60)
    print("BILAN DES ENTRAÎNEMENTS E3-K")
    print("=" * 60)

    print(
        "Combinaisons k/graine disponibles :",
        len(completed_test),
        "/18"
    )

    print("\nNombre de graines par valeur de k :")

    print(
        completed_test.groupby("k")["seed"]
        .nunique()
        .to_string()
    )

    if len(completed_test) == 18:

        print("\nTous les 18 entraînements sont terminés.")

    else:

        print(
            "\nCertains entraînements restent à effectuer."
        )

print("\nCellule 4 exécutée.")

## 5. Contrôle de cohérence k=2 et tableau de l'article

Le résultat historique de `G_operational` est **RMSE 0.619072 ± 0.053739 °C** (3 graines). Si le nouveau résultat `k=2` diffère, le notebook affiche cet écart et il convient de présenter l'expérience E3-K comme un protocole complémentaire homogène, plutôt que comme une stricte réplication bit à bit des entraînements précédents.

In [ ]:
# ============================================================
# E3-K — Contrôle de cohérence k=2 et tableau de l'article
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Chargement des résultats d'entraînement
# ------------------------------------------------------------

csv_path = RESULT_DIR / "03_per_seed_all_splits.csv"

if not csv_path.exists():
    raise RuntimeError(
        "Aucun CSV d'entraînement trouvé. "
        "Activez RUN_TRAINING et exécutez la cellule 4."
    )

runs = pd.read_csv(csv_path)

# Conserver uniquement les résultats de test
tests = runs[runs["split"] == "test"].copy()

if tests.empty:
    raise RuntimeError(
        "Aucun résultat de test trouvé dans le fichier CSV."
    )

# Éviter les doublons éventuels après plusieurs exécutions
tests = tests.drop_duplicates(
    subset=["k", "seed"],
    keep="last"
)

# ------------------------------------------------------------
# 2. Calcul des statistiques par valeur de k
# ------------------------------------------------------------

summary = (
    tests.groupby("k")
    .agg(
        n_seeds=("seed", "nunique"),
        RMSE_mean=("RMSE", "mean"),
        RMSE_SD=("RMSE", "std"),
        MAE_mean=("MAE", "mean"),
        MAE_SD=("MAE", "std"),
        R2_mean=("R2", "mean")
    )
    .reset_index()
)

# Ajouter les caractéristiques des graphes
summary = topology_table.merge(
    summary,
    on="k",
    how="left"
)

# ------------------------------------------------------------
# 3. Mise en forme des résultats
# ------------------------------------------------------------

def format_rmse(row):

    if (
        pd.notna(row["n_seeds"])
        and row["n_seeds"] == 3
    ):
        return (
            f'{row["RMSE_mean"]:.4f} ± '
            f'{row["RMSE_SD"]:.4f}'
        )

    return "INCOMPLET"


summary["Test RMSE (°C)"] = summary.apply(
    format_rmse,
    axis=1
)

summary.to_csv(
    RESULT_DIR / "04_k_results_mean_SD.csv",
    index=False
)

print("\n=== Résultats de sensibilité du GCN ===\n")

display(
    summary[
        [
            "k",
            "edges",
            "density_percent",
            "n_seeds",
            "Test RMSE (°C)"
        ]
    ]
)

# ------------------------------------------------------------
# 4. Comparaison k=2 avec l'expérience originale
# ------------------------------------------------------------

hist = original_metrics.query(
    "topology == 'G_operational' and split == 'test'"
)

old_mean = hist["RMSE"].mean()
old_sd = hist["RMSE"].std(ddof=1)

k2 = summary[summary["k"] == 2]

if not k2.empty and pd.notna(k2["RMSE_mean"].iloc[0]):

    new_mean = float(k2["RMSE_mean"].iloc[0])
    new_sd = float(k2["RMSE_SD"].iloc[0])

    difference = new_mean - old_mean

    print("\n=== Contrôle de reproductibilité k=2 ===")

    print(
        f"GCN original : "
        f"{old_mean:.6f} ± {old_sd:.6f} °C"
    )

    print(
        f"GCN réentraîné : "
        f"{new_mean:.6f} ± {new_sd:.6f} °C"
    )

    print(
        f"Différence RMSE moyen : "
        f"{difference:+.6f} °C"
    )

    if abs(difference) > 0.05:
        print(
            "\nATTENTION : écart supérieur à 0.05 °C."
            "\nVérifier la procédure d'entraînement."
        )
    else:
        print(
            "\nContrôle k=2 : écart moyen inférieur à 0.05 °C."
            "\nCela ne garantit pas une reproduction exacte."
        )

# ------------------------------------------------------------
# 5. Vérification des 18 entraînements
# ------------------------------------------------------------

completed = int(
    (summary["n_seeds"] == 3).sum()
)

total = len(K_VALUES)

print("\n=== État des entraînements ===")

print(
    f"Configurations complètes : {completed}/{total}"
)

print(
    f"Entraînements de test disponibles : "
    f"{len(tests)}/18"
)

if completed == total:

    print(
        "\nTous les entraînements sont terminés."
    )

    print(
        "Les résultats peuvent être utilisés "
        "pour compléter le tableau 6A."
    )

else:

    missing_k = summary.loc[
        summary["n_seeds"].fillna(0) < 3,
        "k"
    ].tolist()

    print(
        "\nConfigurations encore incomplètes :",
        missing_k
    )

    print(
        "Terminez les entraînements correspondants "
        "avant l'interprétation scientifique."
    )

## 6. Export pour le manuscrit

Le tableau est autonome, et les prédictions test sont conservées pour d'éventuelles comparaisons statistiques appariées. **Ne pas sélectionner k à partir du test** : cette analyse est une analyse de sensibilité descriptive et non une optimisation d'hyperparamètre. Les graphes reflètent des **similarités opérationnelles**, pas des liens physiques vérifiés.

In [ ]:
# ============================================================
# E3-K — Export des résultats pour le manuscrit
# ============================================================

import json
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. Vérification des entraînements
# ============================================================

if summary["n_seeds"].fillna(0).eq(3).all():
    print("Les 18 entraînements sont disponibles.")
    complete = True
else:
    print("ATTENTION : certains entraînements sont incomplets.")
    print("Les résultats seront exportés à titre provisoire.")
    complete = False

# ============================================================
# 2. Tableau 6A pour l'article
# ============================================================

article = summary[
    ["k", "edges", "density_percent", "Test RMSE (°C)"]
].copy()

article.columns = [
    "k",
    "Edges",
    "Density (%)",
    "Test RMSE (°C)"
]

article["Density (%)"] = article["Density (%)"].map(
    lambda a: f"{a:.1f}"
)

article.to_csv(
    RESULT_DIR / "05_TABLE_6A_for_article.csv",
    index=False
)

print("\n=== TABLE 6A ===\n")
print(article.to_string(index=False))

# ============================================================
# 3. Figure de sensibilité
# ============================================================

# Ne représenter que les configurations ayant trois graines
plot_data = summary[
    summary["n_seeds"] == 3
].copy()

if not plot_data.empty:

    plt.figure(figsize=(7.5, 4.5))

    plt.errorbar(
        plot_data["k"],
        plot_data["RMSE_mean"],
        yerr=plot_data["RMSE_SD"],
        fmt="o-",
        capsize=4
    )

    plt.xticks(K_VALUES)

    plt.xlabel("Nearest-neighbor parameter k")
    plt.ylabel("Test RMSE (°C), mean ± SD")

    plt.title(
        "Sensitivity of GCN performance to graph connectivity"
    )

    plt.grid(alpha=0.25)
    plt.tight_layout()

    plt.savefig(
        RESULT_DIR / "06_k_sensitivity_RMSE.png",
        dpi=180
    )

    plt.show()

else:
    print("Aucune configuration complète à représenter.")

# ============================================================
# 4. Manifeste de reproductibilité
# ============================================================

manifest = {
    "experiment": "E3-K GCN connectivity sensitivity",
    "source": str(OUTPUTS_SOURCE),
    "seed_list": SEEDS,
    "k_values": K_VALUES,
    "completed": complete,
    "architecture": (
        "2-layer primary GCN, hidden=64, "
        "ReLU, 0.1 dropout, linear head"
    ),
    "adjacency": (
        "Unweighted undirected union of train-only "
        "top-k Pearson with self-loops "
        "and symmetric normalization"
    ),
    "input_source": (
        "07D_final_alignment/X_graph_*_scaled.npy"
    ),
    "target_source": (
        "07D_final_alignment/y_graph_*.npy"
    ),
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "optimizer": "Adam",
    "lr": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "dropout_training_note": (
        "Training dropout ordering documented in notebook; "
        "original archive does not contain source training code."
    ),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "device": str(DEVICE),
    "checkpoint_inference_validation": (
        checks_df.to_dict(orient="records")
    )
}

manifest_path = RESULT_DIR / "07_protocol_and_validation.json"

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
        default=str
    ),
    encoding="utf-8"
)

print("\nFichiers enregistrés dans :", RESULT_DIR.resolve())

if complete:
    print("Tableau 6A complet : prêt pour vérification scientifique.")
else:
    print(
        "Tableau 6A provisoire : "
        "terminer les entraînements avant intégration dans l'article."
    )